# فيديو ستيكمان سريع مع صوت عربي أو إنجليزي
شخصية كرتونية بتعابير وإيماءات، عناوين وعناصر توضيحية وصوت AI محلي. الرسم يعمل على CPU. الصوت الافتراضي عصبي عبر الإنترنت؛ ويمكن اختيار Piper للعمل محليًا.

**لأول تجربة سريعة اختر `reviewed_example`**: سكربت مراجع جاهز. اختر `idea` لكتابة السكربت بالـAI، ويمكن تشغيله على CPU أو GPU اختياريًا. تثبيت المكتبات وتنزيل أوزان النص في وضع idea أول مرة قد يستغرق عدة دقائق؛ لا يتكرر لكل فيديو داخل الجلسة.

القياسات المحلية موثّقة في `docs/FAST_PRESENTER.md`. جودة النص والنطق تحتاج المراجعة. الصوت الافتراضي يستخدم Microsoft Edge عبر الإنترنت، وليس أوزانًا صوتية مدرّبة داخل المشروع. لا يتحول إلى Piper عند فشل الخدمة. الموسيقى الأصلية والمؤثرات مفعّلة مع خفض الموسيقى أثناء الكلام. استنساخ XTTS خيار محلي أبطأ.

In [ ]:
from pathlib import Path
import zipfile
import os
import subprocess
import sys
from google.colab import files, drive

# Download Code -> Download ZIP from your private GitHub repository, then upload it here.
# Set True to upload newer code in an existing Colab session.
UPDATE_SOURCE = False
if UPDATE_SOURCE or not Path('/content/Voice2Reel/voice2reel/presenter.py').exists():
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one repository ZIP from GitHub')
    import tempfile
    destination = Path(tempfile.mkdtemp(prefix='v2r-upload-', dir='/content'))
    with zipfile.ZipFile(archives[0]) as archive:
        if sum(info.file_size for info in archive.infolist()) > 50_000_000:
            raise ValueError('Expected source-only repository ZIP, under 50 MB')
        for info in archive.infolist():
            if not (destination / info.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(destination)
    projects = list(destination.rglob('pyproject.toml'))
    if len(projects) != 1:
        raise ValueError('Cannot identify repository root')
    import shutil
    shutil.copytree(projects[0].parent, '/content/Voice2Reel', dirs_exist_ok=True)
    del uploaded
os.chdir('/content/Voice2Reel')
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/Voice2Reel')
WORK.mkdir(parents=True, exist_ok=True)
print('Persistent artifacts:', WORK)


## اختر وضع التشغيل

In [ ]:
MODE = "reviewed_example" #@param ["reviewed_example", "idea"]
PROFILE = "fast" #@param ["fast", "quality"]
VOICE_ENGINE = "edge" #@param ["edge", "piper"]
SPEECH_RATE = 0 #@param {type:"slider", min:-20, max:25, step:1}
MUSIC = True #@param {type:"boolean"}
TRANSITION_SFX = True #@param {type:"boolean"}

AI_DEVICE = "auto" #@param ["auto", "cpu", "cuda"]


## تثبيت مرة واحدة لكل جلسة

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.8'], check=True)
if not Path('/content/v2r-presenter/bin/python').is_file():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '/content/v2r-presenter'], check=True)
PY = '/content/v2r-presenter/bin/python'
def pip_install(*args):
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PY, *args], check=True)
def run(*args):
    subprocess.run([PY, *map(str, args)], check=True)
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'fonts-dejavu-core'], check=True)
pip_install('-r', 'requirements/presenter.txt')
DEVICE = 'cpu'
if MODE == 'idea':
    import shutil
    has_gpu = False
    if shutil.which('nvidia-smi'):
        check = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
        has_gpu = check.returncode == 0 and 'GPU' in check.stdout
    DEVICE = ('cuda' if has_gpu else 'cpu') if AI_DEVICE == 'auto' else AI_DEVICE
    if DEVICE == 'cuda':
        if not has_gpu:
            raise RuntimeError('Select a GPU runtime, or choose AI_DEVICE=cpu')
        pip_install('torch==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126')
        pip_install('-r', 'requirements/text.txt')
    else:
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'build-essential'], check=True)
        os.environ['CC'] = 'gcc'
        os.environ['CXX'] = 'g++'
        os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=OFF -DLLAMA_CURL=OFF'
        os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] = '2'
        pip_install('--no-binary', 'llama-cpp-python', '-r', 'requirements/director-cpu.txt')
    print('Text model device:', DEVICE)
freeze = subprocess.check_output([sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True)
(WORK / 'fast_presenter_environment.txt').write_text(freeze)


## الفكرة وخمسة أسئلة
في وضع `reviewed_example` يُستخدم سكربت المثال الموافق للغة؛ الفكرة والإجابات الأخرى تخص وضع `idea`.

In [ ]:
IDEA = "ثلاث نصائح لتعلّم البرمجة بشكل منظّم" #@param {type:"string"}
AUDIENCE = "مبتدئون" #@param {type:"string"}
DURATION = 20 #@param {type:"slider", min:10, max:60, step:1}
LANGUAGE = "ar" #@param ["ar", "en"]
FACTS = "ابدأ بموضوع واحد، طبّق بمثال صغير، وراجع ما تعلمته. لا تعد بنتائج مضمونة ولا تخترع أرقامًا." #@param {type:"string"}
VISUAL_FOCUS = "شرح بثلاث خطوات واضحة" #@param {type:"string"}


## إنتاج الفيديو
المقاس السريع 540×960 و24 FPS. تغيير المشاهد أو المقاس مع ثبات الكلام يعيد الرسم فقط. إعادة الطلب المطابق تستخدم النتائج المحفوظة.

In [ ]:
PROJECT = WORK / 'presenter_v2'
args = ['-m', 'voice2reel.presenter', '--output', PROJECT, '--profile', PROFILE,
        '--voice-dir', WORK / 'models' / 'piper', '--voice-engine', VOICE_ENGINE]
if VOICE_ENGINE == 'edge':
    args += ['--speech-rate', SPEECH_RATE]
if not MUSIC:
    args += ['--no-music']
if not TRANSITION_SFX:
    args += ['--no-sfx']
if MODE == 'reviewed_example':
    example = 'examples/presenter_v2/semicolon_ar.json' if LANGUAGE == 'ar' else 'examples/fast_presenter/semicolon_en.json'
    args += ['--script', example, '--device', 'cpu']
else:
    args += ['--idea', IDEA, '--facts', FACTS, '--audience', AUDIENCE,
             '--language', LANGUAGE, '--duration', DURATION, '--visual-focus', VISUAL_FOCUS,
             '--device', DEVICE]
run(*args)
print((PROJECT / 'metrics.json').read_text())


## شاهد واسمع وراجع النص قبل النشر

In [ ]:
from IPython.display import Video, Audio, display
display(Video(str(PROJECT / 'video.mp4'), embed=True))
display(Audio(str(PROJECT / 'soundtrack.wav')))
print((PROJECT / 'script.json').read_text())
files.download(str(PROJECT / 'video.mp4'))


## اختياري: تعديل السكربت وإعادة الإخراج
عدّل `script.json` في Drive. يمكن تغيير الإيماءات والتعابير والعناصر، أو تصحيح الكلام. شغّل الخلية التالية. إذا تغيّر الكلام سيعاد توليد الصوت، وإلا يُستخدم الصوت السابق.

In [ ]:
edit_args = ['-m', 'voice2reel.presenter', '--script', PROJECT / 'script.json', '--output', PROJECT,
             '--profile', PROFILE, '--voice-dir', WORK / 'models' / 'piper', '--device', 'cpu',
             '--voice-engine', VOICE_ENGINE]
if VOICE_ENGINE == 'edge':
    edit_args += ['--speech-rate', SPEECH_RATE]
if not MUSIC:
    edit_args += ['--no-music']
if not TRANSITION_SFX:
    edit_args += ['--no-sfx']
run(*edit_args)
display(Video(str(PROJECT / 'video.mp4'), embed=True))
print((PROJECT / 'metrics.json').read_text())
